# KSE-30 Hybrid Model: XGBoost + LSTM
### Multi-factor Stock Return Prediction — Pakistan Equity Market
---
**Data:** 29 KSE-30 stocks | 12 features | 52,002 samples | 2018–2025

**Pipeline:**
1. Setup & Load
2. Train / Val / Test Split + 60-Day Window Construction
3. XGBoost — Train & Evaluate
4. LSTM — Train & Evaluate
5. Hybrid Ensemble (Optimal Weighted Average)
6. Evaluation Figures (300 DPI)
7. Save All Outputs

> ⚡ **Enable GPU before running:**
> Runtime → Change runtime type → T4 GPU → Save

## 0. Install & Import

In [ ]:
!pip install xgboost scikit-learn pandas numpy matplotlib seaborn tensorflow -q
print('Done.')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import glob, pickle, warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb

import tensorflow as tf
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from matplotlib.patches import Patch

# ── Global plot settings — 300 DPI ───────────────────────────────────────────
plt.rcParams['figure.dpi']     = 300
plt.rcParams['savefig.dpi']    = 300
plt.rcParams['axes.grid']      = True
plt.rcParams['grid.alpha']     = 0.3
plt.rcParams['font.size']      = 9
sns.set_style('whitegrid')

SEED   = 42
WINDOW = 60
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('TF version    :', tf.__version__)
print('GPU available :', len(tf.config.list_physical_devices('GPU')) > 0)
print('XGBoost       :', xgb.__version__)

---
## 1. Load Data

In [ ]:
from google.colab import files
print('Upload: KSE30_model_ready.csv')
uploaded = files.upload()
fname = list(uploaded.keys())[0]

df = pd.read_csv(fname)
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(['Ticker','Date']).reset_index(drop=True)

# Column definitions — based on actual file structure
META_COLS    = ['Date', 'Ticker', 'Sector']
TARGET_COL   = 'Forward_Log_Return'
FEATURE_COLS = [c for c in df.columns if c not in META_COLS + [TARGET_COL]]
N_FEATURES   = len(FEATURE_COLS)

# Clip extreme target outliers beyond 5 std to stabilise LSTM training
tgt_mean = df[TARGET_COL].mean()
tgt_std  = df[TARGET_COL].std()
df[TARGET_COL] = df[TARGET_COL].clip(tgt_mean - 5*tgt_std, tgt_mean + 5*tgt_std)

print(f'Shape          : {df.shape}')
print(f'Stocks         : {df["Ticker"].nunique()}')
print(f'Date range     : {df["Date"].min().date()} -> {df["Date"].max().date()}')
print(f'Features ({N_FEATURES})  : {FEATURE_COLS}')
print(f'Target range   : [{df[TARGET_COL].min():.4f}, {df[TARGET_COL].max():.4f}]')
print(f'Missing        : {df.isnull().sum().sum()}')

---
## 2. Train / Val / Test Split + 60-Day Window Construction
| Split | Period | Purpose |
|---|---|---|
| Train | Jan 2018 – Dec 2023 | Model learning |
| Validation | Jan 2024 – Jun 2024 | Hyperparameter tuning + early stopping |
| Test | Jul 2024 – Dec 2025 | Final unbiased evaluation |

In [ ]:
TRAIN_END = '2023-12-31'
VAL_END   = '2024-06-30'

df_train = df[df['Date'] <= TRAIN_END].copy()
df_val   = df[(df['Date'] > TRAIN_END) & (df['Date'] <= VAL_END)].copy()
df_test  = df[df['Date'] > VAL_END].copy()

print(f'Train : {df_train["Date"].min().date()} -> {df_train["Date"].max().date()}  ({len(df_train):,} rows)')
print(f'Val   : {df_val["Date"].min().date()}   -> {df_val["Date"].max().date()}    ({len(df_val):,} rows)')
print(f'Test  : {df_test["Date"].min().date()}  -> {df_test["Date"].max().date()}   ({len(df_test):,} rows)')

In [ ]:
def make_windows(data, feature_cols, target_col, window=60):
    """
    Convert time-series data into supervised learning windows.
    X_seq  : (n, window, n_features)      LSTM input
    X_flat : (n, window * n_features)     XGBoost input
    y      : (n,)                         next-day log return
    meta   : DataFrame Date/Ticker/Sector per sample
    """
    X_seq_list, X_flat_list, y_list, meta_list = [], [], [], []
    for ticker, grp in data.groupby('Ticker'):
        grp = grp.sort_values('Date').reset_index(drop=True)
        if len(grp) <= window:
            continue
        feat   = grp[feature_cols].values.astype(np.float32)
        tgt    = grp[target_col].values.astype(np.float32)
        dates  = grp['Date'].values
        sector = grp['Sector'].iloc[0]
        for i in range(window, len(grp)):
            w = feat[i-window:i]
            X_seq_list.append(w)
            X_flat_list.append(w.flatten())
            y_list.append(tgt[i])
            meta_list.append({'Date': dates[i], 'Ticker': ticker, 'Sector': sector})
    return (
        np.array(X_seq_list,  dtype=np.float32),
        np.array(X_flat_list, dtype=np.float32),
        np.array(y_list,      dtype=np.float32),
        pd.DataFrame(meta_list)
    )

print('Building 60-day windows...')
X_seq_tr, X_flat_tr, y_tr, meta_tr = make_windows(df_train, FEATURE_COLS, TARGET_COL, WINDOW)
X_seq_va, X_flat_va, y_va, meta_va = make_windows(df_val,   FEATURE_COLS, TARGET_COL, WINDOW)
X_seq_te, X_flat_te, y_te, meta_te = make_windows(df_test,  FEATURE_COLS, TARGET_COL, WINDOW)

print(f'Train : X_seq={X_seq_tr.shape}  X_flat={X_flat_tr.shape}  y={y_tr.shape}')
print(f'Val   : X_seq={X_seq_va.shape}  X_flat={X_flat_va.shape}  y={y_va.shape}')
print(f'Test  : X_seq={X_seq_te.shape}  X_flat={X_flat_te.shape}  y={y_te.shape}')

In [ ]:
def evaluate(y_true, y_pred, label=''):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    da   = np.mean(np.sign(y_true) == np.sign(y_pred)) * 100
    if label:
        print(f'  {label:<8}  RMSE={rmse:.6f}  MAE={mae:.6f}  R2={r2:+.4f}  DA={da:.2f}%')
    return {'RMSE': rmse, 'MAE': mae, 'R2': r2, 'DA': da}

---
## 3. XGBoost Model

In [ ]:
print('Training XGBoost...')
xgb_model = xgb.XGBRegressor(
    n_estimators=600, max_depth=5, learning_rate=0.03,
    subsample=0.75, colsample_bytree=0.75,
    min_child_weight=10, gamma=0.2,
    reg_alpha=0.1, reg_lambda=2.0,
    random_state=SEED, n_jobs=-1,
    eval_metric='rmse', early_stopping_rounds=40, verbosity=0
)
xgb_model.fit(X_flat_tr, y_tr, eval_set=[(X_flat_va, y_va)], verbose=100)

xgb_pr_tr = xgb_model.predict(X_flat_tr)
xgb_pr_va = xgb_model.predict(X_flat_va)
xgb_pr_te = xgb_model.predict(X_flat_te)

print('\n--- XGBoost ---')
xgb_m_tr = evaluate(y_tr, xgb_pr_tr, 'Train')
xgb_m_va = evaluate(y_va, xgb_pr_va, 'Val')
xgb_m_te = evaluate(y_te, xgb_pr_te, 'Test')

In [ ]:
# Figure 1 — XGBoost Learning Curve
val_rmse = xgb_model.evals_result()['validation_0']['rmse']
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(val_rmse, color='#e74c3c', lw=1.2, label='Validation RMSE')
ax.axvline(xgb_model.best_iteration, color='black', ls='--', lw=1,
           label=f'Best iteration = {xgb_model.best_iteration}')
ax.set_title('XGBoost — Validation RMSE over Boosting Rounds', fontsize=11, fontweight='bold')
ax.set_xlabel('Boosting Round')
ax.set_ylabel('RMSE')
ax.legend()
plt.tight_layout()
plt.savefig('fig_model_01_XGB_LearningCurve.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: fig_model_01_XGB_LearningCurve.png')

In [ ]:
# Figure 2 — XGBoost Feature Importance (aggregated across 60-day window)
raw_imp = xgb_model.feature_importances_
agg_imp = {f: 0.0 for f in FEATURE_COLS}
idx = 0
for t in range(WINDOW):
    for f in FEATURE_COLS:
        agg_imp[f] += raw_imp[idx]
        idx += 1

imp_df = pd.DataFrame(list(agg_imp.items()), columns=['Feature','Importance'])
imp_df = imp_df.sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(12, 7))
colors  = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(imp_df)))
ax.barh(imp_df['Feature'], imp_df['Importance'], color=colors, edgecolor='white')
ax.set_title('XGBoost — Aggregated Feature Importance (Summed across 60-day window)',
             fontsize=11, fontweight='bold')
ax.set_xlabel('Total Importance Score')
for i, val in enumerate(imp_df['Importance']):
    ax.text(val + 0.0001, i, f'{val:.4f}', va='center', fontsize=7)
plt.tight_layout()
plt.savefig('fig_model_02_XGB_FeatureImportance.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: fig_model_02_XGB_FeatureImportance.png')

---
## 4. LSTM Model

In [ ]:
def build_lstm(window, n_features):
    inp = Input(shape=(window, n_features), name='input')
    x = LSTM(128, return_sequences=True,  name='lstm_1')(inp)
    x = BatchNormalization()(x)
    x = Dropout(0.25)(x)
    x = LSTM(64,  return_sequences=True,  name='lstm_2')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.25)(x)
    x = LSTM(32,  return_sequences=False, name='lstm_3')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.15)(x)
    x = Dense(16, activation='relu')(x)
    x = Dropout(0.10)(x)
    out = Dense(1, name='output')(x)
    model = Model(inputs=inp, outputs=out)
    model.compile(
        optimizer=Adam(learning_rate=0.001, clipnorm=1.0),
        loss=tf.keras.losses.Huber(delta=0.5),
        metrics=['mae']
    )
    return model

lstm_model = build_lstm(WINDOW, N_FEATURES)
lstm_model.summary()

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6, verbose=1),
    ModelCheckpoint('lstm_best.keras', monitor='val_loss', save_best_only=True, verbose=0)
]

history = lstm_model.fit(
    X_seq_tr, y_tr,
    validation_data=(X_seq_va, y_va),
    epochs=100, batch_size=512,
    callbacks=callbacks, verbose=1
)

lstm_model = load_model('lstm_best.keras')
lstm_pr_tr = lstm_model.predict(X_seq_tr, verbose=0).flatten()
lstm_pr_va = lstm_model.predict(X_seq_va, verbose=0).flatten()
lstm_pr_te = lstm_model.predict(X_seq_te, verbose=0).flatten()

print('\n--- LSTM ---')
lstm_m_tr = evaluate(y_tr, lstm_pr_tr, 'Train')
lstm_m_va = evaluate(y_va, lstm_pr_va, 'Val')
lstm_m_te = evaluate(y_te, lstm_pr_te, 'Test')

In [ ]:
# Figure 3 — LSTM Training History
ep = range(1, len(history.history['loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].plot(ep, history.history['loss'],     color='#3498db', lw=1.2, label='Train')
axes[0].plot(ep, history.history['val_loss'], color='#e74c3c', lw=1.2, label='Val')
axes[0].set_title('LSTM — Huber Loss', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].legend()

axes[1].plot(ep, history.history['mae'],     color='#3498db', lw=1.2, label='Train')
axes[1].plot(ep, history.history['val_mae'], color='#e74c3c', lw=1.2, label='Val')
axes[1].set_title('LSTM — MAE', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('MAE'); axes[1].legend()

plt.suptitle('LSTM Training History', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_model_03_LSTM_History.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: fig_model_03_LSTM_History.png')

---
## 5. Hybrid Ensemble

In [ ]:
alphas    = np.arange(0.0, 1.01, 0.05)
rmse_grid = []
da_grid   = []

for a in alphas:
    h = a * xgb_pr_va + (1-a) * lstm_pr_va
    rmse_grid.append(np.sqrt(mean_squared_error(y_va, h)))
    da_grid.append(np.mean(np.sign(y_va) == np.sign(h)) * 100)

best_alpha = alphas[np.argmin(rmse_grid)]
print(f'Optimal weights → XGBoost: {best_alpha:.2f}   LSTM: {1-best_alpha:.2f}')

hyb_pr_tr = best_alpha * xgb_pr_tr + (1-best_alpha) * lstm_pr_tr
hyb_pr_va = best_alpha * xgb_pr_va + (1-best_alpha) * lstm_pr_va
hyb_pr_te = best_alpha * xgb_pr_te + (1-best_alpha) * lstm_pr_te

print('\n--- Hybrid ---')
hyb_m_tr = evaluate(y_tr, hyb_pr_tr, 'Train')
hyb_m_va = evaluate(y_va, hyb_pr_va, 'Val')
hyb_m_te = evaluate(y_te, hyb_pr_te, 'Test')

---
## 6. All Evaluation Figures (300 DPI)

In [ ]:
# Figure 4 — Ensemble Weight Search
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].plot(alphas, rmse_grid, color='#e74c3c', lw=1.5, marker='o', ms=4)
axes[0].axvline(best_alpha, color='black', ls='--', lw=1, label=f'Optimal α={best_alpha:.2f}')
axes[0].set_title('Ensemble Weight Search — Validation RMSE', fontweight='bold')
axes[0].set_xlabel('XGBoost Weight (α)   LSTM Weight = (1−α)')
axes[0].set_ylabel('RMSE'); axes[0].legend()

axes[1].plot(alphas, da_grid, color='#2ecc71', lw=1.5, marker='o', ms=4)
axes[1].axvline(best_alpha, color='black', ls='--', lw=1, label=f'Optimal α={best_alpha:.2f}')
axes[1].axhline(50, color='grey', ls=':', lw=1, label='Random (50%)')
axes[1].set_title('Ensemble Weight Search — Directional Accuracy', fontweight='bold')
axes[1].set_xlabel('XGBoost Weight (α)   LSTM Weight = (1−α)')
axes[1].set_ylabel('DA (%)'); axes[1].legend()

plt.suptitle('Hybrid Ensemble — Weight Optimisation on Validation Set', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_model_04_EnsembleSearch.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: fig_model_04_EnsembleSearch.png')

In [ ]:
# Figure 5 — Model Comparison Bar Chart
models  = ['XGBoost', 'LSTM', 'Hybrid']
clrs    = ['#3498db', '#e74c3c', '#2ecc71']
metrics = {
    'RMSE (lower=better)'        : [xgb_m_te['RMSE'],  lstm_m_te['RMSE'],  hyb_m_te['RMSE']],
    'MAE (lower=better)'         : [xgb_m_te['MAE'],   lstm_m_te['MAE'],   hyb_m_te['MAE']],
    'R² Score (higher=better)'   : [xgb_m_te['R2'],    lstm_m_te['R2'],    hyb_m_te['R2']],
    'Directional Accuracy %'     : [xgb_m_te['DA'],    lstm_m_te['DA'],    hyb_m_te['DA']],
}
fig, axes = plt.subplots(1, 4, figsize=(22, 6))
for ax, (title, vals) in zip(axes, metrics.items()):
    bars = ax.bar(models, vals, color=clrs, edgecolor='white', width=0.5)
    ax.set_title(title, fontsize=9, fontweight='bold')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()*1.01,
                f'{v:.4f}', ha='center', fontsize=8, fontweight='bold')
plt.suptitle('Model Performance Comparison — Test Set (Jul 2024 – Dec 2025)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_model_05_ModelComparison.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: fig_model_05_ModelComparison.png')

In [ ]:
# Figure 6 — Predicted vs Actual Scatter
clip = 0.15
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, preds, label, color in zip(axes,
    [xgb_pr_te, lstm_pr_te, hyb_pr_te],
    ['XGBoost','LSTM','Hybrid'],
    ['#3498db','#e74c3c','#2ecc71']):
    yt = np.clip(y_te,  -clip, clip)
    yp = np.clip(preds, -clip, clip)
    ax.scatter(yt, yp, alpha=0.08, s=3, color=color, rasterized=True)
    ax.plot([-clip,clip],[-clip,clip],'k--',lw=1,label='Perfect')
    r2 = r2_score(y_te, preds)
    da = np.mean(np.sign(y_te)==np.sign(preds))*100
    ax.set_title(f'{label}\nR²={r2:.4f}   DA={da:.1f}%', fontweight='bold')
    ax.set_xlabel('Actual Log Return')
    ax.set_ylabel('Predicted Log Return')
    ax.legend(fontsize=7)
plt.suptitle('Predicted vs Actual Log Returns — Test Set', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_model_06_PredVsActual.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: fig_model_06_PredVsActual.png')

In [ ]:
# Figure 7 — Residuals Distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, preds, label, color in zip(axes,
    [xgb_pr_te, lstm_pr_te, hyb_pr_te],
    ['XGBoost','LSTM','Hybrid'],
    ['#3498db','#e74c3c','#2ecc71']):
    res = y_te - preds
    ax.hist(res.clip(-0.10, 0.10), bins=80, color=color,
            edgecolor='white', alpha=0.85, density=True)
    ax.axvline(0,         color='black', ls='--', lw=1)
    ax.axvline(res.mean(),color='red',   ls='-',  lw=1, label=f'Mean={res.mean():.5f}')
    ax.set_title(f'{label} — Residuals', fontweight='bold')
    ax.set_xlabel('Residual (Actual − Predicted)')
    ax.set_ylabel('Density'); ax.legend(fontsize=8)
plt.suptitle('Residual Distributions — Test Set', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_model_07_Residuals.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: fig_model_07_Residuals.png')

In [ ]:
# Figure 8 — Per-Stock Performance
meta_te = meta_te.copy()
meta_te['Actual'] = y_te
meta_te['Hybrid'] = hyb_pr_te
meta_te['XGB']    = xgb_pr_te
meta_te['LSTM_P'] = lstm_pr_te

per_stock = meta_te.groupby('Ticker').apply(
    lambda g: pd.Series({
        'DA_%'   : np.mean(np.sign(g['Actual'])==np.sign(g['Hybrid']))*100,
        'RMSE'   : np.sqrt(mean_squared_error(g['Actual'],g['Hybrid'])),
        'Samples': len(g)
    })
).reset_index().sort_values('DA_%', ascending=False)

print('=== Per-Stock Hybrid Performance (Test) ==='); print(per_stock.round(4).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(22, 7))
da_colors = ['#2ecc71' if v>=55 else '#e67e22' if v>=50 else '#e74c3c' for v in per_stock['DA_%']]
axes[0].bar(per_stock['Ticker'].str.replace('.KA',''), per_stock['DA_%'],
            color=da_colors, edgecolor='white')
axes[0].axhline(50, color='black', ls='--', lw=1)
axes[0].axhline(per_stock['DA_%'].mean(), color='blue', ls='-.', lw=1,
                label=f'Mean={per_stock["DA_%"].mean():.1f}%')
axes[0].set_title('Directional Accuracy % per Stock — Hybrid (Test)', fontweight='bold')
axes[0].set_ylabel('DA (%)')
axes[0].tick_params(axis='x', rotation=90, labelsize=7)
axes[0].legend(handles=[
    Patch(facecolor='#2ecc71',label='≥55% Strong'),
    Patch(facecolor='#e67e22',label='50–55% Moderate'),
    Patch(facecolor='#e74c3c',label='<50% Below random')
], fontsize=7)

ps_r = per_stock.sort_values('RMSE')
rc   = plt.cm.RdYlGn_r(np.linspace(0.1,0.9,len(ps_r)))
axes[1].bar(ps_r['Ticker'].str.replace('.KA',''), ps_r['RMSE'], color=rc, edgecolor='white')
axes[1].set_title('RMSE per Stock — Hybrid (Test, sorted)', fontweight='bold')
axes[1].set_ylabel('RMSE')
axes[1].tick_params(axis='x', rotation=90, labelsize=7)

plt.suptitle('Per-Stock Performance — Hybrid Ensemble (Test Set)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_model_08_PerStock.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: fig_model_08_PerStock.png')

In [ ]:
# Figure 9 — Time-Series Predictions — 6 Stocks
showcase = ['HBL.KA','OGDC.KA','LUCK.KA','FFC.KA','SYS.KA','MEBL.KA']
fig, axes = plt.subplots(3, 2, figsize=(20, 15))
axes = axes.flatten()
for i, ticker in enumerate(showcase):
    grp = meta_te[meta_te['Ticker']==ticker].sort_values('Date')
    ax  = axes[i]
    ax.plot(pd.to_datetime(grp['Date']), grp['Actual'], color='#2c3e50', lw=0.9, label='Actual')
    ax.plot(pd.to_datetime(grp['Date']), grp['Hybrid'], color='#e74c3c', lw=0.9, label='Hybrid')
    ax.axhline(0, color='black', lw=0.4)
    da   = np.mean(np.sign(grp['Actual'])==np.sign(grp['Hybrid']))*100
    rmse = np.sqrt(mean_squared_error(grp['Actual'],grp['Hybrid']))
    ax.set_title(f'{ticker.replace(".KA","")}  DA={da:.1f}%  RMSE={rmse:.5f}',
                 fontweight='bold', fontsize=9)
    ax.set_ylabel('Log Return')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.tick_params(axis='x', rotation=30, labelsize=7)
    ax.legend(fontsize=7)
plt.suptitle('Hybrid — Predicted vs Actual Log Returns  |  Test: Jul 2024–Dec 2025',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_model_09_TimeSeries.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: fig_model_09_TimeSeries.png')

In [ ]:
# Figure 10 — Metrics Heatmap
rows = []
for model, triplet in [
    ('XGBoost',(xgb_m_tr,xgb_m_va,xgb_m_te)),
    ('LSTM',   (lstm_m_tr,lstm_m_va,lstm_m_te)),
    ('Hybrid', (hyb_m_tr,hyb_m_va,hyb_m_te)),
]:
    for split, m in zip(['Train','Val','Test'], triplet):
        rows.append({'Model':model,'Split':split,
                     'RMSE':round(m['RMSE'],6),'MAE':round(m['MAE'],6),
                     'R2':round(m['R2'],4),'DA_%':round(m['DA'],2)})

metrics_df = pd.DataFrame(rows)
print('=== Complete Metrics ===')
print(metrics_df.to_string(index=False))

p_da   = metrics_df.pivot(index='Model',columns='Split',values='DA_%')[['Train','Val','Test']]
p_rmse = metrics_df.pivot(index='Model',columns='Split',values='RMSE')[['Train','Val','Test']]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.heatmap(p_da,   annot=True, fmt='.2f', cmap='RdYlGn',
            vmin=45, vmax=65, ax=axes[0], linewidths=0.5, annot_kws={'size':12})
axes[0].set_title('Directional Accuracy % across Splits', fontweight='bold')
sns.heatmap(p_rmse, annot=True, fmt='.5f', cmap='RdYlGn_r',
            ax=axes[1], linewidths=0.5, annot_kws={'size':12})
axes[1].set_title('RMSE across Splits', fontweight='bold')
plt.suptitle('Model Performance Heatmap', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_model_10_MetricsHeatmap.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: fig_model_10_MetricsHeatmap.png')

---
## 7. Save All Outputs

In [ ]:
# Predictions CSV — this feeds directly into Black-Litterman
pred_df = meta_te.copy()
pred_df['XGB_Pred']    = xgb_pr_te
pred_df['LSTM_Pred']   = lstm_pr_te
pred_df['Hybrid_Pred'] = hyb_pr_te
pred_df['Actual']      = y_te
pred_df['Dir_Correct'] = (np.sign(y_te)==np.sign(hyb_pr_te)).astype(int)
pred_df.to_csv('KSE30_predictions.csv', index=False)

metrics_df.to_csv('model_metrics_summary.csv', index=False)
per_stock.to_csv('per_stock_metrics.csv',       index=False)

xgb_model.save_model('xgb_model.json')
lstm_model.save('lstm_model.keras')

with open('ensemble_config.pkl','wb') as f:
    pickle.dump({'xgb_weight':float(best_alpha),'lstm_weight':float(1-best_alpha)}, f)

print('All files saved.')

In [ ]:
from google.colab import files
import glob

print('--- Downloading figures (300 DPI) ---')
for f in sorted(glob.glob('fig_model_*.png')):
    files.download(f); print(f'  ✓ {f}')

print('\n--- Downloading data & model files ---')
for f in ['KSE30_predictions.csv','model_metrics_summary.csv',
          'per_stock_metrics.csv','xgb_model.json',
          'lstm_model.keras','ensemble_config.pkl']:
    files.download(f); print(f'  ✓ {f}')

print('\nAll done!')

---
## ✅ Complete

**Key output for next step:** `KSE30_predictions.csv`

This contains `Hybrid_Pred` (predicted next-day log return per stock) which becomes the ML-generated **view** in the Black-Litterman model.
The per-stock Directional Accuracy becomes the **confidence level** for each view.

**Next → Black-Litterman + Portfolio Optimization**